# 03 — Streaming Across Providers: Examples

> Same streaming loop — different providers, normalized by LiteLLM.

**What you'll build:**
1. OpenAI streaming (baseline)
2. Anthropic streaming with event types
3. Unified streaming via LiteLLM
4. Side-by-side benchmark across providers

In [ ]:
import os, time
from dotenv import load_dotenv
from openai import OpenAI
from rich.table import Table
from rich.console import Console
from rich import print as rprint
import litellm

load_dotenv()
litellm.set_verbose = False
client = OpenAI()
console = Console()
print('✅ Setup complete')

---
## Part 1: OpenAI Streaming (Baseline)

In [ ]:
def openai_stream(prompt: str) -> dict:
    start = time.perf_counter()
    ttft = None
    text = ""
    stream = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        stream=True, max_tokens=100
    )
    for chunk in stream:
        c = chunk.choices[0].delta.content or ""
        if c:
            if ttft is None: ttft = time.perf_counter()
            text += c
            print(c, end="", flush=True)
    print()
    return {"provider": "openai/gpt-4o-mini", "text": text,
            "ttft_ms": round((ttft - start)*1000, 1) if ttft else None,
            "total_ms": round((time.perf_counter()-start)*1000, 1)}

print("🟢 OpenAI streaming:\n")
r1 = openai_stream("Name 3 programming languages invented before 1980.")
rprint(f"\n   TTFT: {r1['ttft_ms']}ms | Total: {r1['total_ms']}ms")

---
## Part 2: Anthropic Streaming

In [ ]:
# Uncomment if you have an ANTHROPIC_API_KEY
# import anthropic
# ant_client = anthropic.Anthropic()
#
# def anthropic_stream(prompt: str) -> dict:
#     start = time.perf_counter()
#     ttft = None
#     text = ""
#     with ant_client.messages.stream(
#         model="claude-3-5-haiku-20241022",
#         max_tokens=100,
#         messages=[{"role": "user", "content": prompt}]
#     ) as stream:
#         for t in stream.text_stream:
#             if ttft is None: ttft = time.perf_counter()
#             text += t
#             print(t, end="", flush=True)
#     print()
#     final = stream.get_final_message()
#     return {"provider": "anthropic/claude-haiku", "text": text,
#             "ttft_ms": round((ttft - start)*1000, 1),
#             "total_ms": round((time.perf_counter()-start)*1000, 1),
#             "input_tokens": final.usage.input_tokens,
#             "output_tokens": final.usage.output_tokens}
#
# print("\n🟠 Anthropic streaming:\n")
# r2 = anthropic_stream("Name 3 programming languages invented before 1980.")

print("ℹ️  Anthropic streaming demo — set ANTHROPIC_API_KEY and uncomment above cell")

---
## Part 3: LiteLLM Unified Streaming

In [ ]:
def litellm_stream(prompt: str, model: str) -> dict:
    """Stream any model through LiteLLM — identical code."""
    start = time.perf_counter()
    ttft = None
    text = ""
    try:
        stream = litellm.completion(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            stream=True, max_tokens=80
        )
        for chunk in stream:
            c = chunk.choices[0].delta.content or ""
            if c:
                if ttft is None: ttft = time.perf_counter()
                text += c
                print(c, end="", flush=True)
        print()
        return {"provider": model, "text": text,
                "ttft_ms": round((ttft - start)*1000, 1) if ttft else None,
                "total_ms": round((time.perf_counter()-start)*1000, 1),
                "success": True}
    except Exception as e:
        return {"provider": model, "success": False, "error": str(e)[:60]}

prompt = "Name 3 programming languages invented before 1980."

# These all use IDENTICAL Python code — only model name changes
models = [
    "gpt-4o-mini",
    # "claude-3-5-haiku-20241022",   # Uncomment if Anthropic key available
    # "gemini/gemini-1.5-flash",     # Uncomment if Google key available
]

results = []
for model in models:
    rprint(f"\n[cyan]🔵 {model}:[/cyan]")
    r = litellm_stream(prompt, model)
    results.append(r)

# ── Comparison table ───────────────────────────────────────────────────────
table = Table(title="Provider Streaming Comparison", show_header=True)
table.add_column("Model", style="cyan")
table.add_column("TTFT", style="yellow")
table.add_column("Total", style="white")
table.add_column("Status", style="green")

for r in results:
    if r.get("success"):
        table.add_row(r["provider"], f"{r['ttft_ms']}ms", f"{r['total_ms']}ms", "✅")
    else:
        table.add_row(r["provider"], "-", "-", f"❌ {r.get('error', 'error')}")

console.print(table)

---
## Summary

| Provider | Library | Stream Toggle | Content Access |
|---|---|---|---|
| OpenAI | `openai` | `stream=True` | `chunk.choices[0].delta.content` |
| Anthropic | `anthropic` | `.stream()` CM | `stream.text_stream` |
| Gemini | `google-generativeai` | `stream=True` | `chunk.text` |
| Any | `litellm` | `stream=True` | `chunk.choices[0].delta.content` |

**Next:** `04_streaming_in_agent_loops` — stream while running multi-step agent reasoning.